# Day 55 · Exercise 4: FastAPI Auth Dependency

**What you'll build:** Implement `get_current_user` (a FastAPI `Depends` function) and `build_protected_api()`. This is how auth becomes a one-liner on any route: `def handler(user = Depends(get_current_user))`.

## Setup (provided)

In [ ]:
import bcrypt as _bcrypt_lib
from jose import jwt, JWTError
from datetime import datetime, timedelta
from typing import Annotated
from fastapi import FastAPI, Depends, HTTPException
from fastapi.security import HTTPBearer, HTTPAuthorizationCredentials
from starlette.testclient import TestClient

SECRET_KEY  = "test-secret-for-exercise"
ALGORITHM   = "HS256"

def hash_password(password: str) -> str:
    return _bcrypt_lib.hashpw(password.encode(), _bcrypt_lib.gensalt()).decode()

def verify_password(plain: str, hashed: str) -> bool:
    return _bcrypt_lib.checkpw(plain.encode(), hashed.encode())

def create_token(data: dict, expires_in_minutes: int = 60) -> str:
    payload = data.copy()
    payload["exp"] = datetime.utcnow() + timedelta(minutes=expires_in_minutes)
    return jwt.encode(payload, SECRET_KEY, algorithm=ALGORITHM)

def decode_token(token: str) -> dict:
    return jwt.decode(token, SECRET_KEY, algorithms=[ALGORITHM])

_security = HTTPBearer()


## Your Implementation

In [ ]:
def get_current_user(
    creds: Annotated[HTTPAuthorizationCredentials, Depends(_security)]
) -> dict:
    """FastAPI dependency: validate Bearer JWT and return the decoded payload.

    Args:
        creds: Injected by HTTPBearer — .credentials is the raw token string.
    Returns:
        Decoded JWT payload dict.
    Raises:
        HTTPException(401): If the token is invalid or expired.
    """
    # TODO: try decode_token(creds.credentials); on JWTError raise HTTPException(401, ...)
    raise NotImplementedError

def build_protected_api() -> FastAPI:
    """Build a FastAPI app with one protected route GET /me.

    Returns:
        FastAPI app where GET /me requires a valid Bearer token
        and returns {"user_id": ..., "email": ...}.
    """
    app = FastAPI()
    # TODO: add GET /me using Depends(get_current_user) that returns
    #       {"user_id": user["user_id"], "email": user["email"]}
    return app


In [ ]:
def get_current_user(
    creds: Annotated[HTTPAuthorizationCredentials, Depends(_security)]
) -> dict:
    try:
        return decode_token(creds.credentials)
    except JWTError:
        raise HTTPException(status_code=401, detail="Invalid or expired token")

def build_protected_api() -> FastAPI:
    app = FastAPI()

    @app.get("/me")
    def me(user: dict = Depends(get_current_user)):
        return {"user_id": user["user_id"], "email": user["email"]}

    return app


## Check Your Work

In [ ]:
def _run_checks():
    score = 0
    total = 5

    def _chk(n, ok, msg):
        nonlocal score
        print(f"  {'✅' if ok else '❌'} Check {n}: {msg}")
        if ok:
            score += 1

    try:
        app = build_protected_api()
    except NotImplementedError:
        for i in range(1, total + 1):
            print(f"  ❌ Check {i}: build_protected_api not implemented")
        print(f"\nScore: 0 / {total}")
        return
    except Exception as e:
        for i in range(1, total + 1):
            print(f"  ❌ Check {i}: build_protected_api raised {type(e).__name__}: {e}")
        print(f"\nScore: 0 / {total}")
        return

    client = TestClient(app, raise_server_exceptions=False)

    # check 1: no token → 401 (HTTPBearer auto-rejects missing header)
    r = client.get("/me")
    _chk(1, r.status_code == 401,
         f"no auth header → 401 (got {r.status_code})")

    # check 2: bad token → 401
    r = client.get("/me", headers={"Authorization": "Bearer bad.token.here"})
    _chk(2, r.status_code == 401,
         f"invalid token → 401 (got {r.status_code})")

    # valid token for remaining checks
    good_tok = create_token({"user_id": 99, "email": "bob@example.com"})
    r = client.get("/me", headers={"Authorization": f"Bearer {good_tok}"})
    _chk(3, r.status_code == 200,
         f"valid token → 200 (got {r.status_code})")

    if r.status_code == 200:
        data = r.json()
        _chk(4, data.get("user_id") == 99,
             f"user_id == 99 (got {data.get('user_id')})")
        _chk(5, data.get("email") == "bob@example.com",
             f"email correct (got {data.get('email')})")
    else:
        for i in range(4, 6):
            print(f"  ❌ Check {i}: skipped (check 3 failed)")

    print(f"\nScore: {score} / {total}")
    if score == total:
        print("🎉 Exercise complete!")

_run_checks()


## Bonus Challenge

Add a second protected route `GET /admin` that reads `user['email']` and raises `HTTPException(403, 'admin only')` unless the email ends with `@admin.example.com`. Test it with TestClient using tokens for both admin and regular users. This is the start of role-based access control.

## Solution

<details>
<summary>Show solution</summary>

```python
def get_current_user(
    creds: Annotated[HTTPAuthorizationCredentials, Depends(_security)]
) -> dict:
    try:
        return decode_token(creds.credentials)
    except JWTError:
        raise HTTPException(status_code=401, detail="Invalid or expired token")

def build_protected_api() -> FastAPI:
    app = FastAPI()

    @app.get("/me")
    def me(user: dict = Depends(get_current_user)):
        return {"user_id": user["user_id"], "email": user["email"]}

    return app
```

**Why this works:** `HTTPBearer` is a FastAPI security scheme that reads the
`Authorization: Bearer <token>` header. When the header is missing or malformed it
auto-raises 401; your code runs only when a Bearer token is present. `Depends()`
tells FastAPI to call `get_current_user` and inject its return value into the route
handler — so the handler receives the decoded user dict directly, with no HTTP
machinery visible.

</details>